In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
from sklearn.metrics import classification_report

In [34]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [35]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [36]:
df.nunique()

customerID          7043
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64

In [37]:
print(df["TotalCharges"].dtype)

object


In [38]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors = "coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["TotalCharges"] = df["TotalCharges"].round().astype(int)

In [39]:
df["MonthlyCharges"] = df["MonthlyCharges"].astype(int)

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        7043 non-null   object
 1   gender            7043 non-null   object
 2   SeniorCitizen     7043 non-null   int64 
 3   Partner           7043 non-null   object
 4   Dependents        7043 non-null   object
 5   tenure            7043 non-null   int64 
 6   PhoneService      7043 non-null   object
 7   MultipleLines     7043 non-null   object
 8   InternetService   7043 non-null   object
 9   OnlineSecurity    7043 non-null   object
 10  OnlineBackup      7043 non-null   object
 11  DeviceProtection  7043 non-null   object
 12  TechSupport       7043 non-null   object
 13  StreamingTV       7043 non-null   object
 14  StreamingMovies   7043 non-null   object
 15  Contract          7043 non-null   object
 16  PaperlessBilling  7043 non-null   object
 17  PaymentMethod 

In [41]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29,30,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56,1890,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53,108,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42,1841,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70,152,Yes


In [42]:
x = df.drop(columns = "Churn")
y = df.Churn

In [43]:
num_cols = x.select_dtypes(include = "number").columns
cat_cols = x.select_dtypes(include = "object").columns

In [44]:
xtrain , xtest, ytrain, ytest = train_test_split(x,y,random_state = 42, train_size = 0.8)

In [45]:
cat_cols.nunique()

16

In [46]:
cat_cols

Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [47]:
preprocessing = ColumnTransformer(
    transformers = [
        ("scaler",StandardScaler(),num_cols),
        ("onehot",OneHotEncoder(),['gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod']),
        ("target",TargetEncoder(),"customerID")
    ]
)

In [48]:
main_pipe = Pipeline(
    steps = [
        ("pre",preprocessing),
        ("model",KNeighborsClassifier())
    ]
)

In [49]:
main_pipe.fit(xtrain,ytrain)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('pre', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scaler', ...), ('onehot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers 

In [50]:
grid_cv = GridSearchCV(
    estimator = main_pipe,
    param_grid = {
        "model__n_neighbors" : [3,7,25,27,29,38],
        "model__metric" : ["manhattan","euclidean"]
    },
    cv = 5,
    n_jobs = -1,
    verbose = 1
)

In [51]:
grid_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\bthar\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan        nan        nan        nan
 0.73908631 0.76890218 0.78966999 0.78753981 0.78789553 0.78807094]
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__metric': ['manhattan', 'euclidean'], 'model__n_neighbors': [3, 7, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is

In [52]:
results = grid_cv.cv_results_
pd.DataFrame(results)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__metric,param_model__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.063968,0.010613,0.023645,0.005427,manhattan,3,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
1,0.074889,0.005476,0.021476,0.004114,manhattan,7,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
2,0.069752,0.007906,0.017424,0.001606,manhattan,25,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
3,0.069519,0.007746,0.018662,0.001647,manhattan,27,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
4,0.075770,0.013067,0.023049,0.005845,manhattan,29,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
5,0.073236,0.006780,0.020927,0.003928,manhattan,38,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
6,0.068144,0.012166,0.081445,0.009227,euclidean,3,"{'model__metric': 'euclidean', 'model__n_neigh...",0.741792,0.739130,0.736469,0.726708,0.751332,0.739086,0.007967,6
7,0.087125,0.003242,0.101465,0.007424,euclidean,7,"{'model__metric': 'euclidean', 'model__n_neigh...",0.776398,0.789707,0.751553,0.763088,0.763766,0.768902,0.013040,5
8,0.073688,0.012290,0.106721,0.004935,euclidean,25,"{'model__metric': 'euclidean', 'model__n_neigh...",0.789707,0.790594,0.793256,0.784383,0.790409,0.789670,0.002906,1
9,0.085277,0.010930,0.114824,0.004369,euclidean,27,"{'model__metric': 'euclidean', 'model__n_neigh...",0.791482,0.793256,0.790594,0.775510,0.786856,0.787540,0.006368,4


In [53]:
grid_cv.best_params_

{'model__metric': 'euclidean', 'model__n_neighbors': 25}

In [54]:
main_pipe2 = Pipeline(
    steps = [
        ("pre",preprocessing),
        ("model",KNeighborsRegressor())
    ]
)

In [55]:
grid_cv2 = GridSearchCV(
    estimator = main_pipe2,
    param_grid = {
        "model__n_neighbors" : [3,7,25,27,29,38],
        "model__metric" : ["manhattan","euclidean"]
    },
    cv = 5,
    n_jobs = -1,
    verbose = 1
)

In [56]:
grid_cv2.fit(xtrain,ytrain)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\bthar\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__metric': ['manhattan', 'euclidean'], 'model__n_neighbors': [3, 7, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is

In [57]:
results2 = grid_cv2.cv_results_
pd.DataFrame(results2)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__metric,param_model__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.049795,0.001332,0.306732,0.041549,manhattan,3,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,0.067358,0.006276,0.350587,0.036395,manhattan,7,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,0.086482,0.005940,0.440325,0.055156,manhattan,25,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,0.106128,0.008225,0.486049,0.040838,manhattan,27,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,0.112593,0.016881,0.451828,0.058430,manhattan,29,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
5,0.108320,0.015163,0.414386,0.047409,manhattan,38,"{'model__metric': 'manhattan', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
6,0.121205,0.005691,0.150829,0.020142,euclidean,3,"{'model__metric': 'euclidean', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
7,0.126830,0.017291,0.138603,0.017970,euclidean,7,"{'model__metric': 'euclidean', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
8,0.115439,0.010448,0.155366,0.012511,euclidean,25,"{'model__metric': 'euclidean', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
9,0.129459,0.008061,0.153352,0.017753,euclidean,27,"{'model__metric': 'euclidean', 'model__n_neigh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [58]:
grid_cv2.best_params_

{'model__metric': 'manhattan', 'model__n_neighbors': 3}

In [59]:
ytrain_pred = grid_cv.predict(xtrain)
print(classification_report(ytrain,ytrain_pred))

              precision    recall  f1-score   support

          No       0.87      0.89      0.88      4138
         Yes       0.68      0.62      0.64      1496

    accuracy                           0.82      5634
   macro avg       0.77      0.75      0.76      5634
weighted avg       0.82      0.82      0.82      5634



In [61]:
ytest_pred = grid_cv.predict(xtest)
print(classification_report(ytest,ytest_pred))

              precision    recall  f1-score   support

          No       0.86      0.89      0.87      1036
         Yes       0.66      0.59      0.62       373

    accuracy                           0.81      1409
   macro avg       0.76      0.74      0.75      1409
weighted avg       0.81      0.81      0.81      1409



In [ ]:
ytrain_pred = grid_cv.predict(xtrain)
print(classification_report(ytrain,ytrain_pred))